In [1]:
from pathlib import Path
import pandas as pd
from hemmelig import data_path_2021

p = Path(data_path_2021)

# Initiell datautforsk

In [2]:
info = sorted([e.name[:-6].split("_") + [e.name] for e in p.iterdir()])
df = pd.DataFrame(info, columns=["nettside", "språk", "format", "filnavn"])
df

,nettside,språk,format,filnavn
0,113.no,nob,html,113.no_nob_html.jsonl
1,1700-tallet.no,nob,html,1700-tallet.no_nob_html.jsonl
2,22julisenteret.no,nob,html,22julisenteret.no_nob_html.jsonl
3,aho.no,nno,doc,aho.no_nno_doc.jsonl
4,aho.no,nno,html,aho.no_nno_html.jsonl
...,...,...,...,...
1448,yrkesfisker.no,nno,html,yrkesfisker.no_nno_html.jsonl
1449,yrkesfisker.no,nno,pdf,yrkesfisker.no_nno_pdf.jsonl
1450,yrkesfisker.no,nob,doc,yrkesfisker.no_nob_doc.jsonl
1451,yrkesfisker.no,nob,html,yrkesfisker.no_nob_html.jsonl


In [3]:
antall_filer, antall_unike_nettsider = len(df), len(list(df.groupby("nettside")))
fler_språk = 0
nynorsk = 0
nynorsk_flerspråk = 0
nynorsk_og_bokmål = 0

for nettside, df_ in df.groupby("nettside"):
    unike_språk = set(df_.språk)
    if len(unike_språk) > 1:
        fler_språk += 1
    if "nno" in unike_språk:
        nynorsk += 1
    if len(unike_språk) > 1 and "nno" in unike_språk:
        nynorsk_flerspråk += 1
    if "nno" in unike_språk and "nob" in unike_språk:
        nynorsk_og_bokmål += 1

In [4]:
print(f"""
    Det er {antall_filer} filer i målfrid_data
    Det er {antall_unike_nettsider} unike nettsider
    Der er {fler_språk} nettsider som har fler språk 
    Der er {nynorsk} nettsider som har nynorsk 
    Det er {nynorsk_flerspråk} nettsider som har nynorsk og et annet språk
    Det er {nynorsk_og_bokmål} nettsider som har nynorsk og bokmål
""")


    Det er 1453 filer i målfrid_data
    Det er 483 unike nettsider
    Der er 283 nettsider som har fler språk 
    Der er 289 nettsider som har nynorsk 
    Det er 283 nettsider som har nynorsk og et annet språk
    Det er 283 nettsider som har nynorsk og bokmål



In [5]:
from collections import defaultdict, Counter

df_path = Path("output/dokumenter_statistikk.csv")

if not df_path.exists():
    dok_data = defaultdict(Counter)
    for nettside, df_ in df.groupby("nettside"):    
        for språk, df__ in df_.groupby("språk"):
            filer = df__.filnavn
            for fil in filer:
                filp = p / fil
                jsonfil = pd.read_json(filp, lines=True)
                assert all(jsonfil["lang"] == språk)
                dok_data[nettside][språk] += len(jsonfil)

    dok_data_df = pd.DataFrame([(nettside, counter["nno"], counter["nob"]) for nettside, counter in dok_data.items()], columns=["nettside", "antall_nno_dokumenter", "antall_nob_dokumenter"])
    dok_data_df.to_csv(df_path, index=False)
else:
    dok_data_df = pd.read_csv("output/dokumenter_statistikk.csv")
dok_data_df

,nettside,antall_nno_dokumenter,antall_nob_dokumenter
0,113.no,0,71
1,1700-tallet.no,0,1
2,22julisenteret.no,0,1
3,aho.no,42,1769
4,akademiskskriving.no,0,1
...,...,...,...
478,volven.no,0,1
479,whocc.no,0,3
480,workinnorway.no,0,22
481,yr.no,1,5


### Sidene med flest bokmåls-tekster

In [8]:
dok_data_df.sort_values("antall_nob_dokumenter", ascending=False)[:30]

,nettside,antall_nno_dokumenter,antall_nob_dokumenter
435,uio.no,15024,190263
415,statsforvalteren.no,37793,77910
389,sikt.no,888,63171
434,uib.no,11450,48592
105,forskning.no,1853,42149
411,ssb.no,5395,37668
311,ntnu.no,3602,32490
326,oslomet.no,4101,26914
216,landbruksforum.no,15952,25161
396,skatteetaten.no,1172,21080


### Sidene med flest nynorsk-tekster

In [9]:
dok_data_df.sort_values("antall_nno_dokumenter", ascending=False)[:30]

,nettside,antall_nno_dokumenter,antall_nob_dokumenter
415,statsforvalteren.no,37793,77910
216,landbruksforum.no,15952,25161
435,uio.no,15024,190263
149,hivolda.no,12069,3331
434,uib.no,11450,48592
411,ssb.no,5395,37668
313,nve.no,4218,17237
159,hvl.no,4140,7358
326,oslomet.no,4101,26914
311,ntnu.no,3602,32490
